In [7]:
import pandas as pd
import re

# ================= IEEE =================
def parse_ieee(texte):
    articles = texte.split("\n\n")
    data = []

    for article in articles:
        if not article.strip():
            continue

        lignes = article.strip().split("\n")
        ligne_principale = lignes[0]

        # Auteurs
        auteurs = ligne_principale.split('",')[0]

        # Titre
        title_match = re.search(r'"(.*?)"', ligne_principale)
        titre = title_match.group(1) if title_match else ""

        # Journal
        journal_match = re.search(r'in (.*?), vol', ligne_principale)
        journal = journal_match.group(1) if journal_match else ""

        # Année
        annee_match = re.search(r', (\d{4}),', ligne_principale)
        annee = annee_match.group(1) if annee_match else ""

        # DOI
        doi_match = re.search(r'doi: ([^\.\n]+)', ligne_principale)
        doi = doi_match.group(1) if doi_match else ""

        # Mots-clés
        mots_cles = ""
        if len(lignes) > 1:
            kw_match = re.search(r'keywords: {(.*?)}', lignes[1])
            if kw_match:
                mots_cles = kw_match.group(1)

        data.append({
            "Title": titre,
            "Authors": auteurs,
            "Year": annee,
            "Journal": journal,
            "Abstract": "",
            "DOI": doi,
            "Keywords": mots_cles,
            "Source_DB": "IEEE"
        })

    return pd.DataFrame(data)


# ================= GOOGLE SCHOLAR =================
import re
import pandas as pd

def parse_google_scholar(texte):
    data = []

    # Séparation intelligente (format [1], [2], etc.)
    blocks = re.split(r"\n\s*\[\d+\]\s*", texte)

    for block in blocks:
        if not block.strip():
            continue

        # On ignore les blocs sans titre
        if "Title:" not in block:
            continue

        # ================= EXTRACTION =================
        titre = re.search(r'Title:\s*(.*)', block)
        auteurs = re.search(r'Authors?:\s*(.*)', block)
        annee = re.search(r'Year:\s*(\d{4})', block)
        source = re.search(r'Source:\s*(.*)', block)
        resume = re.search(r'Abstract:\s*(.*)', block, re.S)

        # ================= CLEAN =================
        title = titre.group(1).strip() if titre else ""
        authors = auteurs.group(1).strip() if auteurs else ""
        year = annee.group(1) if annee else ""
        journal = source.group(1).strip() if source else ""
        abstract = resume.group(1).strip() if resume else ""

        # Nettoyage abstract (coupe si trop long ou bruit)
        abstract = re.split(r'\n[A-Z ]+:', abstract)[0].strip()

        # ================= AJOUT =================
        data.append({
            "Title": title,
            "Authors": authors,
            "Year": year,
            "Journal": journal,
            "Abstract": abstract,
            "DOI": None,  # IMPORTANT (pas "")
            "Keywords": "",
            "Source_DB": "Google Scholar"
        })

    df = pd.DataFrame(data)

    # Nettoyage titres (important pour déduplication)
    if not df.empty:
        df["Title"] = (
            df["Title"]
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    return df


# ================= SCOPUS =================
def parse_scopus(texte):
    articles = re.split(r'\n(?=[A-Z][a-zA-Z\-]+.*,)', texte)
    data = []

    for article in articles:
        if "DOI" not in article:
            continue

        lignes = article.strip().split("\n")

        auteurs = lignes[0].strip()
        titre = lignes[2].strip() if len(lignes) > 2 else ""

        annee = ""
        journal = ""

        match = re.search(r'\((\d{4})\)\s*(.*)', article)
        if match:
            annee = match.group(1)
            journal = match.group(2).split(",")[0]

        doi_match = re.search(r'DOI:\s*(\S+)', article)
        doi = doi_match.group(1) if doi_match else ""

        abstract_match = re.search(
            r'ABSTRACT:\s*(.*?)(?:AUTHOR KEYWORDS:|INDEX KEYWORDS:)',
            article,
            re.S
        )
        abstract = abstract_match.group(1).strip() if abstract_match else ""

        keywords_match = re.search(r'AUTHOR KEYWORDS:\s*(.*)', article)
        keywords = keywords_match.group(1).strip() if keywords_match else ""

        data.append({
            "Title": titre,
            "Authors": auteurs,
            "Year": annee,
            "Journal": journal,
            "Abstract": abstract,
            "DOI": doi,
            "Keywords": keywords,
            "Source_DB": "Scopus"
        })

    return pd.DataFrame(data)

In [8]:
with open("downloads/scopus_export_Mar 30-2026_31c9e799-c6ca-4a69-bbd8-3bd2a75ca1fa.txt", "r", encoding="utf-8") as f:
    scopus_text = f.read()

with open("downloads/IEEE Xplore Citation Plain Text Download 2026.3.30.19.20.42.txt", "r", encoding="utf-8") as f:
    ieee_text = f.read()

with open("downloads/google_scholar.txt", "r", encoding="utf-8") as f:
    gs_text = f.read()

In [9]:
df_scopus = parse_scopus(scopus_text)
df_ieee = parse_ieee(ieee_text)
df_gs = parse_google_scholar(gs_text)

df_all = pd.concat([df_scopus, df_ieee, df_gs], ignore_index=True)

In [10]:
!pip install openpyxl

In [13]:
# ================= NORMALIZATION =================

def normalize_doi(doi):
    if not doi:
        return None
    return doi.lower().replace("https://doi.org/", "").strip()

def normalize_title(title):
    if not title:
        return ""
    return (
        title.lower()
        .replace("-", " ")
        .replace(":", " ")
        .replace(".", "")
        .strip()
    )

# Appliquer nettoyage
df_all["DOI"] = df_all["DOI"].apply(normalize_doi)
df_all["Title_clean"] = df_all["Title"].apply(normalize_title)

# ================= DEDUPLICATION =================

# 1. Supprimer doublons avec DOI
df_all = df_all.drop_duplicates(subset=["DOI"], keep="first")

# 2. Séparer
df_with_doi = df_all[df_all["DOI"].notna()]
df_no_doi = df_all[df_all["DOI"].isna()]

# 3. Supprimer doublons sur titres (Google Scholar surtout)
df_no_doi = df_no_doi.drop_duplicates(subset=["Title_clean"], keep="first")

# 4. Recombiner
df_all = pd.concat([df_with_doi, df_no_doi], ignore_index=True)

df_all.to_excel("downloads/RGB-T_sans_doublons.xlsx", index=False)
print("DONE:", len(df_all), "articles")

DONE: 86 articles
